# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display the dataset's name and description
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets (tables), fields (columns), and their `@id`s.
You can use these IDs in later steps for referring to parts of the data schema.

In [ ]:
# List available record sets and their fields with @id

record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set @id: {rs['@id']}")
    print(f"  Name: {rs['name']}")
    print(f"  Description: {rs.get('description', '')}")
    
    # List fields (columns)
    print("  Fields (columns):")
    for field in rs.get('field', []):
        if isinstance(field, dict):  # field is expanded field dict
            field_id = field['@id']
            field_name = field.get('name', '')
        else:  # field is just @id string
            # Try to get full field info from dataset.fields
            try:
                field_info = next(f for f in dataset.fields if f['@id'] == field)
                field_id = field_info['@id']
                field_name = field_info.get('name', '')
            except Exception:
                field_id = field
                field_name = ''
        print(f"    - {field_name} (@id: {field_id})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from above.

In [ ]:
# Choose record set(s) to extract
# From overview, you may see (for example): record_sets[0]['@id'] == 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034--ProteinList'
# and others. Adjust below as needed or print for confirmation.

record_set_ids = [rs['@id'] for rs in record_sets]  # use all record sets for demonstration

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Loaded DataFrame with shape: {dataframes[record_set_id].shape}")
    else:
        print(f"  No records found.")

# Show all record set IDs loaded
print("\nLoaded record set IDs:")
for rsid in dataframes.keys():
    print(f"  - {rsid}")

# For next steps, pick a main protein table. For this dataset it's typically the main record set (largest table):
if dataframes:
    main_record_set_id = max(dataframes, key=lambda x: dataframes[x].shape[0])
    print(f"\nUsing main record set: {main_record_set_id}\n")
    print("Available columns:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No loaded DataFrames!")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA: Filter, normalize, group by attributes
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('mode.chained_assignment', None)

df = dataframes[main_record_set_id]

# Identify numeric fields in main table
numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric fields in main record set:")
print(numeric_fields)

if numeric_fields:
    # Choose the first numeric field for demonstration
    numeric_field = numeric_fields[0]
    threshold = df[numeric_field].mean()  # use mean as example threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalize the chosen field (z-score)
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nFirst 5 normalized values of {numeric_field}:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by potential categorical field (search for suggestive fields)
    potential_group_fields = [col for col in df.columns if df[col].dtype == object and df[col].nunique() < 10]
    if potential_group_fields:
        group_field = potential_group_fields[0]
        print(f"\nGrouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'count'])
        display(grouped_df.head())
    else:
        print("No suitable grouping fields found.")
else:
    print("No numeric fields found in main table.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
if numeric_fields:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    # Boxplot by group field if available
    if 'group_field' in locals():
        plt.figure(figsize=(10,4))
        df.boxplot(column=numeric_field, by=group_field, grid=False)
        plt.title(f'{numeric_field} by {group_field}')
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset provides detailed protein measurements from mass spectrometry on human mast cells, including quantitative and categorical information.
- Data can be loaded, filtered, and normalized using Python and mlcroissant with entity `@id` referencing for reproducibility.
- Typical EDA and visualization operations are readily supported, making this dataset suitable for biomarker discovery, expression analysis, and methodological demonstration.

Continue exploring more derived fields, relationships, and advanced analysis as desired!